In [2]:
import pandas as pd
import numpy as np
import re


df = pd.read_json("../data/raw/Subscription_Boxes.jsonl", lines=True)

print("Initial Shape:", df.shape)
df.head()

Initial Shape: (16216, 10)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1,USELESS,Absolutely useless nonsense and a complete was...,[],B07G584SHG,B09WC47S3V,AEMJ2EG5ODOCYUTI54NBXZHDJGSQ,2020-10-08 05:10:57.705,2,True
1,2,Manufactured where?,"With a couple of the items, I wasn't quite sur...",[],B07QL1JRCN,B07QL1JRCN,AEEJBFZKUBEEMBZUZJV4UHFVEEBQ,2020-12-27 23:12:15.433,20,True
2,1,Little bang for your buck.,Two SMALL stuffed animals and 2 little bags of...,[],B07RBYJN37,B08N5QKX1Y,AGSVZNZBTSGQBKZDZTQYEZHGDPCQ,2021-01-06 12:48:35.319,4,True
3,5,New favorite box,"Although I don’t remember signing up for this,...",[],B07KM6T8GV,B07KM6T8GV,AFDERNB6BIR3U2DOR3S2KX7KJJXQ,2021-03-19 12:19:11.887,1,True
4,5,Coctique,I loved every thing and could use it all. Thin...,[],B07NVL6TJG,B07NVKNVNM,AE6P2YJ6FKX332MD56GPJFSHXNJQ,2019-06-03 03:40:06.066,0,True


In [3]:

df = df.rename(columns={
    "text": "review_text",
    "helpful_vote": "helpful",
    "verified_purchase": "verified"
})

df.columns = [col.lower().strip() for col in df.columns]

df.head()

,rating,title,review_text,images,asin,parent_asin,user_id,timestamp,helpful,verified
0,1,USELESS,Absolutely useless nonsense and a complete was...,[],B07G584SHG,B09WC47S3V,AEMJ2EG5ODOCYUTI54NBXZHDJGSQ,2020-10-08 05:10:57.705,2,True
1,2,Manufactured where?,"With a couple of the items, I wasn't quite sur...",[],B07QL1JRCN,B07QL1JRCN,AEEJBFZKUBEEMBZUZJV4UHFVEEBQ,2020-12-27 23:12:15.433,20,True
2,1,Little bang for your buck.,Two SMALL stuffed animals and 2 little bags of...,[],B07RBYJN37,B08N5QKX1Y,AGSVZNZBTSGQBKZDZTQYEZHGDPCQ,2021-01-06 12:48:35.319,4,True
3,5,New favorite box,"Although I don’t remember signing up for this,...",[],B07KM6T8GV,B07KM6T8GV,AFDERNB6BIR3U2DOR3S2KX7KJJXQ,2021-03-19 12:19:11.887,1,True
4,5,Coctique,I loved every thing and could use it all. Thin...,[],B07NVL6TJG,B07NVKNVNM,AE6P2YJ6FKX332MD56GPJFSHXNJQ,2019-06-03 03:40:06.066,0,True


In [4]:

df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

df[["timestamp", "date"]].head()

,timestamp,date
0,2020-10-08 05:10:57.705,2020-10-08 05:10:57.705
1,2020-12-27 23:12:15.433,2020-12-27 23:12:15.433
2,2021-01-06 12:48:35.319,2021-01-06 12:48:35.319
3,2021-03-19 12:19:11.887,2021-03-19 12:19:11.887
4,2019-06-03 03:40:06.066,2019-06-03 03:40:06.066


In [5]:

print("Missing Values:\n", df.isnull().sum())

df = df.dropna(subset=["review_text", "rating"])

print("After Dropping Missing:", df.shape)

Missing Values:
 rating         0
title          0
review_text    0
images         0
asin           0
parent_asin    0
user_id        0
timestamp      0
helpful        0
verified       0
date           0
dtype: int64
After Dropping Missing: (16216, 11)


In [9]:
before = df.shape[0]
# Convert list/dict columns to strings to make them hashable
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].apply(lambda x: str(x) if isinstance(x, (list, dict)) else x)

df = df.drop_duplicates()
after = df.shape[0]

print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


In [10]:

#Fix Data Types

df["rating"] = df["rating"].astype(float)
df["verified"] = df["verified"].astype(bool)

df.dtypes

rating                float64
title                     str
review_text               str
images                    str
asin                      str
parent_asin               str
user_id                   str
timestamp      datetime64[ms]
helpful                 int64
verified                 bool
date           datetime64[ms]
dtype: object

In [11]:

# Clean Text Data

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # remove special chars
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

df["cleaned_text"] = df["review_text"].apply(clean_text)

df[["review_text", "cleaned_text"]].head()

,review_text,cleaned_text
0,Absolutely useless nonsense and a complete was...,absolutely useless nonsense and a complete was...
1,"With a couple of the items, I wasn't quite sur...",with a couple of the items i wasnt quite sure ...
2,Two SMALL stuffed animals and 2 little bags of...,two small stuffed animals and 2 little bags of...
3,"Although I don’t remember signing up for this,...",although i dont remember signing up for this i...
4,I loved every thing and could use it all. Thin...,i loved every thing and could use it all thing...


In [12]:

# Feature Engineering


# Sentiment from rating
def sentiment(r):
    if r >= 4:
        return "Positive"
    elif r == 3:
        return "Neutral"
    else:
        return "Negative"

df["sentiment"] = df["rating"].apply(sentiment)

# Review length (word count)
df["review_length"] = df["cleaned_text"].apply(lambda x: len(x.split()))

# Helpful votes (fill missing)
df["helpful"] = df["helpful"].fillna(0)

df.head()

,rating,title,review_text,images,asin,parent_asin,user_id,timestamp,helpful,verified,date,cleaned_text,sentiment,review_length
0,1.0,USELESS,Absolutely useless nonsense and a complete was...,[],B07G584SHG,B09WC47S3V,AEMJ2EG5ODOCYUTI54NBXZHDJGSQ,2020-10-08 05:10:57.705,2,True,2020-10-08 05:10:57.705,absolutely useless nonsense and a complete was...,Negative,16
1,2.0,Manufactured where?,"With a couple of the items, I wasn't quite sur...",[],B07QL1JRCN,B07QL1JRCN,AEEJBFZKUBEEMBZUZJV4UHFVEEBQ,2020-12-27 23:12:15.433,20,True,2020-12-27 23:12:15.433,with a couple of the items i wasnt quite sure ...,Negative,28
2,1.0,Little bang for your buck.,Two SMALL stuffed animals and 2 little bags of...,[],B07RBYJN37,B08N5QKX1Y,AGSVZNZBTSGQBKZDZTQYEZHGDPCQ,2021-01-06 12:48:35.319,4,True,2021-01-06 12:48:35.319,two small stuffed animals and 2 little bags of...,Negative,27
3,5.0,New favorite box,"Although I don’t remember signing up for this,...",[],B07KM6T8GV,B07KM6T8GV,AFDERNB6BIR3U2DOR3S2KX7KJJXQ,2021-03-19 12:19:11.887,1,True,2021-03-19 12:19:11.887,although i dont remember signing up for this i...,Positive,17
4,5.0,Coctique,I loved every thing and could use it all. Thin...,[],B07NVL6TJG,B07NVKNVNM,AE6P2YJ6FKX332MD56GPJFSHXNJQ,2019-06-03 03:40:06.066,0,True,2019-06-03 03:40:06.066,i loved every thing and could use it all thing...,Positive,19


In [13]:

df = df.drop(columns=["images", "user_id", "parent_asin", "timestamp"], errors="ignore")

df.columns

Index(['rating', 'title', 'review_text', 'asin', 'helpful', 'verified', 'date',
       'cleaned_text', 'sentiment', 'review_length'],
      dtype='str')

In [14]:

# Final Dataset Check

print("Final Shape:", df.shape)
df.head()

Final Shape: (16017, 10)


,rating,title,review_text,asin,helpful,verified,date,cleaned_text,sentiment,review_length
0,1.0,USELESS,Absolutely useless nonsense and a complete was...,B07G584SHG,2,True,2020-10-08 05:10:57.705,absolutely useless nonsense and a complete was...,Negative,16
1,2.0,Manufactured where?,"With a couple of the items, I wasn't quite sur...",B07QL1JRCN,20,True,2020-12-27 23:12:15.433,with a couple of the items i wasnt quite sure ...,Negative,28
2,1.0,Little bang for your buck.,Two SMALL stuffed animals and 2 little bags of...,B07RBYJN37,4,True,2021-01-06 12:48:35.319,two small stuffed animals and 2 little bags of...,Negative,27
3,5.0,New favorite box,"Although I don’t remember signing up for this,...",B07KM6T8GV,1,True,2021-03-19 12:19:11.887,although i dont remember signing up for this i...,Positive,17
4,5.0,Coctique,I loved every thing and could use it all. Thin...,B07NVL6TJG,0,True,2019-06-03 03:40:06.066,i loved every thing and could use it all thing...,Positive,19


In [ ]:

# Save Cleaned Data

df.to_csv("../data/processed/cleaned_reviews.csv", index=False)

print("Cleaned data saved to data/processed/cleaned_reviews.csv")

✅ Cleaned data saved to data/processed/cleaned_reviews.csv
